In [4]:
# Cell 1 — Install packages
!pip install easyocr tqdm openpyxl pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 30.7 MB/s eta 0:00:00


In [5]:
# Cell 2 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import os
# List everything in your MyDrive
os.listdir('/content/drive/MyDrive/')

['GTA V',
 'fifa2022_kit2_1734_20220609_034736_625.JPG',
 'My Documents',
 'IMG-20240925-WA0429.jpg',
 'prothom_alo_crime (draft).gsheet',
 'Colab Notebooks',
 'News_data_Multimodal',
 'Newsdata_Part2',
 'Water Level Indicator.pptx',
 'BCICIV_2a_gdf.zip',
 'eegnet_transfer_project_pytorch.rar',
 'Equipment List (SSS).pdf',
 'Nusaifa turns one',
 'ED Lab Report_01 Group_04 Section_A.gdoc',
 'ED Lab Report_02 Group_04 Section_A.gdoc',
 'ED Lab Report_03 Group_04 Section_A.gdoc',
 'ED Lab Report_04 Group_04 Section_A.gdoc',
 'ED Lab Report_05 Group_04 Section_A.gdoc',
 'Copy of Graduates.gsite',
 'Copy of Student Portfolio.gsite',
 'Copy of Abbott Teacher Template.gsite',
 'Copy of Portal.gsite',
 'Copy of Krona Personal Site Free.gsite',
 'Copy of My Stack + Tier (COPY ME 👉 in 3 Dot Menu).gsite',
 'Copy of Family Update.gsite',
 'Copy of Club.gsite',
 'CV.gdoc',
 'Safid_Hasan_Cover_Letter_Samsung.gdoc',
 '20260316_091302_1.mp4',
 'Memes 2',
 'Portfolio.gsite',
 'facebook_comments.gsheet'

In [8]:
import easyocr
import os
import concurrent.futures
from tqdm import tqdm
import pandas as pd
import threading
import warnings
import re # Add this import for natural sorting

# ---------------- CONFIGURATION ----------------
IMAGE_FOLDER = "/content/drive/MyDrive/Memes 3"   # 🔹 Change to your Drive folder path
OUTPUT_FILE  = "/content/drive/MyDrive/Memes_3.xlsx"  # Saved directly to Drive
LANGUAGES    = ['en', 'bn']
MAX_WORKERS  = 2      # Colab has 2 vCPUs — keep this at 2
SAVE_INTERVAL = 5     # Save every 5 images (reduces Drive I/O)

# ---------------- SILENCE WARNINGS ----------------
warnings.filterwarnings("ignore", message=".*pin_memory.*")
warnings.filterwarnings("ignore", category=UserWarning)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# ---------------- INITIALIZE READER ----------------
print("🔍 Initializing EasyOCR reader (English + Bangla)...")
reader = easyocr.Reader(LANGUAGES, gpu=True)   # Colab free tier gives a T4 GPU
print("🧩 EasyOCR backend:", "GPU ✅" if reader.device == 'cuda' else "CPU ⚠️")

# ---------------- THREAD LOCK ----------------
lock = threading.Lock()

# ---------------- LOAD EXISTING RESULTS (resume support) ----------------
if os.path.exists(OUTPUT_FILE):
    try:
        processed_results_df = pd.read_excel(OUTPUT_FILE)
        print(f"🔁 Resuming — already processed: {len(processed_results_df)} images.")
    except Exception:
        processed_results_df = pd.DataFrame(columns=["image_name", "extracted_text"])
else:
    processed_results_df = pd.DataFrame(columns=["image_name", "extracted_text"])

processed_names = set(processed_results_df["image_name"].tolist())

# ---------------- OCR FUNCTION ----------------
def process_image(image_path):
    file_name = os.path.basename(image_path)
    if file_name in processed_names:
        return None, None, None

    try:
        results = reader.readtext(image_path, detail=1, paragraph=True)
        extracted_text = " ".join([res[1] for res in results])
    except Exception as e:
        extracted_text = f"Error: {str(e)}"

    return image_path, file_name, extracted_text

# ---------------- NATURAL SORT HELPER ----------------
def natural_key(path):
    stem = os.path.splitext(os.path.basename(path))[0]
    # Split the stem into alternating non-digit and digit segments
    # Convert digit segments to integers for numerical comparison
    # Convert non-digit segments to lowercase for case-insensitive alphabetical comparison
    return [int(text) if text.isdigit() else text.lower()
            for text in re.split('([0-9]+)', stem)]

# ---------------- MAIN EXECUTION ----------------
def main():
    all_images = sorted(
        [
            os.path.join(IMAGE_FOLDER, f)
            for f in os.listdir(IMAGE_FOLDER)
            if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))
        ],
        key=natural_key
    )

    to_process = [img for img in all_images if os.path.basename(img) not in processed_names]
    print(f"📂 Total images found : {len(all_images)}")
    print(f"🚀 Images to process  : {len(to_process)}\n")

    new_results = []

    def save_progress():
        """Merge new results with existing ones and write to Drive."""
        temp_df   = pd.DataFrame(new_results, columns=["image_name", "extracted_text"])
        combined  = pd.concat([processed_results_df, temp_df], ignore_index=True)
        # Apply the same natural sorting logic here to prevent TypeError
        combined.sort_values("image_name", key=lambda s: s.map(
            lambda x: [int(text) if text.isdigit() else text.lower()
                       for text in re.split('([0-9]+)', os.path.splitext(x)[0])]
        ), inplace=True)
        combined.to_excel(OUTPUT_FILE, index=False)

    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(process_image, img): img for img in to_process}

        with tqdm(total=len(to_process), desc="🧠 OCR Progress", unit="img") as pbar:
            for future in concurrent.futures.as_completed(futures):
                _, file_name, extracted_text = future.result()
                if file_name is None:
                    pbar.update(1)
                    continue

                pbar.set_postfix_str(f"📄 {file_name[:40]}")
                pbar.update(1)

                with lock:
                    new_results.append((file_name, extracted_text))
                    if len(new_results) % SAVE_INTERVAL == 0:
                        save_progress()

    # Final save
    save_progress()
    print(f"\n✅ Done! Total rows in output: {len(new_results) + len(processed_results_df)}")
    print(f"📁 Saved to: {OUTPUT_FILE}")

# ---------------- ENTRY POINT ----------------
main()

🔍 Initializing EasyOCR reader (English + Bangla)...
🧩 EasyOCR backend: GPU ✅
📂 Total images found : 4447
🚀 Images to process  : 4447



🧠 OCR Progress: 100%|██████████| 4447/4447 [56:03<00:00,  1.32img/s, 📄 4446.jpg]



✅ Done! Total rows in output: 4447
📁 Saved to: /content/drive/MyDrive/Memes_3.xlsx
